# NexaTel Customer Churn Analytics
## Phase 3 — KPI Calculation

### Objective

The objective of this phase is to calculate the key business KPIs required for the NexaTel customer churn analysis.

The KPIs will measure customer churn and retention, revenue performance, customer lifetime value, service performance, contract renewals and customer tenure.

All calculations will use the cleaned datasets generated during Phase 1.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [3]:
cleaned_data_path = Path("../data/cleaned")

WE'LL INITIALLY LOAD THE TABLES MOST LIKELY TO BE REQUIRED:

In [4]:
customers = pd.read_csv(cleaned_data_path / "customers.csv")
subscriptions = pd.read_csv(cleaned_data_path / "subscriptions.csv")
billing = pd.read_csv(cleaned_data_path / "billing.csv")
payments = pd.read_csv(cleaned_data_path / "payments.csv")
contracts = pd.read_csv(cleaned_data_path / "contracts.csv")
support_tickets = pd.read_csv(cleaned_data_path / "support_tickets.csv")

Why these tables?

| Table | Likely KPI use |
|---|---|
| `customers` | Churn, retention, tenure |
| `subscriptions` | ARPU, MRR, CLV |
| `billing` | Revenue-related KPIs |
| `payments` | Payment/revenue analysis |
| `contracts` | Contract renewal |
| `support_tickets` | FCR and resolution time |

In [5]:
print("Customers:", customers.shape)
print("Subscriptions:", subscriptions.shape)
print("Billing:", billing.shape)
print("Payments:", payments.shape)
print("Contracts:", contracts.shape)
print("Support Tickets:", support_tickets.shape)

Customers: (19000, 25)
Subscriptions: (20523, 8)
Billing: (114969, 9)
Payments: (92086, 7)
Contracts: (10607, 9)
Support Tickets: (31490, 11)


In [6]:
print("CUSTOMERS")
for column in customers.columns:
    print(column)

print("\nSUBSCRIPTIONS")
for column in subscriptions.columns:
    print(column)

print("\nBILLING")
for column in billing.columns:
    print(column)

print("\nPAYMENTS")
for column in payments.columns:
    print(column)

print("\nCONTRACTS")
for column in contracts.columns:
    print(column)

print("\nSUPPORT TICKETS")
for column in support_tickets.columns:
    print(column)

CUSTOMERS
customer_id
first_name
last_name
gender
date_of_birth
age
email
phone_number
address
city_id
city_name
state_id
pin_code
city_tier
region_id
occupation
annual_income_inr
customer_segment
product_line
plan_id
acquisition_channel
acquisition_date
tenure_months
customer_status
churn_date

SUBSCRIPTIONS
subscription_id
customer_id
plan_id
start_date
end_date
billing_cycle
monthly_charge_inr
subscription_status

BILLING
invoice_id
customer_id
billing_date
billing_period_month
base_amount_inr
gst_amount_inr
total_amount_inr
due_date
payment_status

PAYMENTS
payment_id
invoice_id
customer_id
payment_date
amount_inr
payment_method
payment_status

CONTRACTS
contract_id
customer_id
plan_id
contract_type
contract_length_months
start_date
end_date
auto_renew
renewal_status

SUPPORT TICKETS
ticket_id
customer_id
created_date
category
priority
channel
status
first_contact_resolution
assigned_employee_id
resolution_hours
resolution_date


## KPI 1: Customer Churn Rate

Churn Rate measures the percentage of customers who have churned out of the total customer base.

Formula:
Churn Rate = (Churned Customers / Total Customers) × 100

In [7]:
total_customers = customers["customer_id"].nunique()

churned_customers = (
    customers["customer_status"] == "Churned"
).sum()

churn_rate = (
    churned_customers / total_customers
) * 100

print("Total Customers:", total_customers)
print("Churned Customers:", churned_customers)
print(f"Churn Rate: {churn_rate:.2f}%")

Total Customers: 19000
Churned Customers: 3506
Churn Rate: 18.45%


## KPI 2: Customer Retention Rate

Retention Rate measures the percentage of customers who have remained active.

Formula:
Retention Rate = (Active Customers / Total Customers) × 100

In [8]:
active_customers = (
    customers["customer_status"] == "Active"
).sum()

retention_rate = (
    active_customers / total_customers
) * 100

print("Total Customers:", total_customers)
print("Active Customers:", active_customers)
print(f"Retention Rate: {retention_rate:.2f}%")

Total Customers: 19000
Active Customers: 15494
Retention Rate: 81.55%


## KPI 3: Average Revenue Per User (ARPU)

ARPU measures the average monthly revenue generated per customer.

For this project, ARPU is calculated using the monthly charge from the latest subscription record for each customer.

Formula:

ARPU = Total Monthly Charges / Total Customers

WE ALREADY HAVE SUBSCRIPTIONS LOADED. BEFORE CALCULATING ARPU, WE NEED TO MAKE SURE WE'RE USING ONE CURRENT SUBSCRIPTION PER CUSTOMER, BECAUSE THE SUBSCRIPTIONS TABLE CAN CONTAIN MULTIPLE RECORDS FOR THE SAME CUSTOMER DUE TO PLAN CHANGES. WE'RE DOING THIS BECAUSE WE DON'T WANT TO ACCIDENTALLY COUNT OLD SUBSCRIPTION PLANS ALONGSIDE THE CUSTOMER'S CURRENT PLAN WHEN CALCULATING ARPU.

STEP 1: CHECK SUBSCRIPTION RECORDS PER CUSTOMER

In [9]:
subscription_counts = (
    subscriptions.groupby("customer_id")
    .size()
)

print("Total subscription records:", len(subscriptions))
print("Unique customers with subscriptions:", subscription_counts.index.nunique())
print("Customers with multiple subscriptions:", (subscription_counts > 1).sum())

Total subscription records: 20523
Unique customers with subscriptions: 19000
Customers with multiple subscriptions: 1523


STEP 2: IDENTIFY THE LATEST SUBSCRIPTION

In [10]:
print("Subscription status values:")
print(subscriptions["subscription_status"].value_counts(dropna=False))

print("\nBilling cycle values:")
print(subscriptions["billing_cycle"].value_counts(dropna=False))

print("\nStart date sample:")
print(subscriptions["start_date"].head(10))

print("\nEnd date sample:")
print(subscriptions["end_date"].head(10))

Subscription status values:
subscription_status
Active        16930
Terminated     3491
NaN             102
Name: count, dtype: int64

Billing cycle values:
billing_cycle
Monthly    15107
Annual      5416
Name: count, dtype: int64

Start date sample:
0    15-01-2026
1    01-01-2014
2    11-11-2015
3    18-10-2014
4    20-10-2014
5    20-06-2018
6    13-01-2024
7    11-09-2024
8    28-10-2014
9    01-02-2026
Name: start_date, dtype: str

End date sample:
0    22-03-2026
1           NaN
2    22-12-2015
3           NaN
4    22-10-2023
5           NaN
6    06-07-2025
7           NaN
8           NaN
9    22-03-2026
Name: end_date, dtype: str


### Why this step matters?
We know from Phase 1 that dates are stored as DD-MM-YYYY text. But before selecting the latest subscription, we'll explicitly convert start_date to a proper date format.



STEP 3: CREATE THE LATEST SUBSCRIPTION RECORD FOR EACH CUSTOMER

In [12]:
subscriptions["start_date_parsed"] = pd.to_datetime(
    subscriptions["start_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

subscriptions = subscriptions.sort_values(
    ["customer_id", "start_date_parsed"]
)

latest_subscriptions = (
    subscriptions
    .drop_duplicates(subset="customer_id", keep="last")
    .copy()
)

print("Latest subscription records:", len(latest_subscriptions))
print("Unique customers:", latest_subscriptions["customer_id"].nunique())

Latest subscription records: 19000
Unique customers: 19000


STEP 4: CHECK THE ACTIVE CUSTOMERS AND MONTHLY CHARGES

In [13]:
active_latest = latest_subscriptions[
    latest_subscriptions["subscription_status"] == "Active"
]

print("Active customers:", len(active_latest))
print("Missing monthly charges:", active_latest["monthly_charge_inr"].isna().sum())
print("\nMonthly charge statistics:")
print(active_latest["monthly_charge_inr"].describe())

Active customers: 15668
Missing monthly charges: 0

Monthly charge statistics:
count    15668.000000
mean       885.124904
std       1395.920800
min         99.000000
25%        300.000000
50%        499.000000
75%        999.000000
max       8333.000000
Name: monthly_charge_inr, dtype: float64


STEP 5: CALCULATE ARPU

In [14]:
total_monthly_revenue = active_latest["monthly_charge_inr"].sum()

active_customers = active_latest["customer_id"].nunique()

arpu = total_monthly_revenue / active_customers

print(f"Total Monthly Revenue: ₹{total_monthly_revenue:,.2f}")
print(f"Active Customers: {active_customers}")
print(f"ARPU: ₹{arpu:,.2f}")

Total Monthly Revenue: ₹13,868,137.00
Active Customers: 15668
ARPU: ₹885.12


## KPI 4: Customer Lifetime Value (CLV)

### Objective
Calculate Customer Lifetime Value using the formula specified in the project requirements.

CLV = ARPU × Gross Margin % × (1 / Churn Rate)

Before calculating CLV, we will check whether a gross margin value is available in the provided datasets.

In [17]:
plans = pd.read_csv(cleaned_data_path / "plans.csv")

In [18]:
print("Customers columns:")
for column in customers.columns:
    print(column)

print("\nPlans columns:")
for column in plans.columns:
    print(column)

Customers columns:
customer_id
first_name
last_name
gender
date_of_birth
age
email
phone_number
address
city_id
city_name
state_id
pin_code
city_tier
region_id
occupation
annual_income_inr
customer_segment
product_line
plan_id
acquisition_channel
acquisition_date
tenure_months
customer_status
churn_date

Plans columns:
plan_id
plan_name
product_line
billing_cycle
monthly_charge
data_quota_gb
voice_minutes
sms_count
segment
is_active


STEP 1: CALCULATE AVERAGE TENURE

In [20]:
average_tenure = customers["tenure_months"].mean()

print(f"Average Customer Tenure: {average_tenure:.2f} months")

Average Customer Tenure: 76.93 months


NOTE: the NexaTel dataset does not contain a cost/direct-cost field or a predefined Gross Margin %. The available revenue-related fields are charges and billing amounts; the data dictionary does not provide the cost needed to calculate gross margin. So we should not invent a gross-margin percentage.

Since the dataset doesn't provide the cost information needed for Gross Margin %, we won't calculate CLV using the gross-margin formula. So, We decided to use the simplified CLV formula:

CLV = ARPU x Average Customer Lifetime


STEP 2: CALCULATE CLV

In [21]:
clv = arpu * average_tenure

print(f"ARPU: ₹{arpu:,.2f}")
print(f"Average Customer Lifetime: {average_tenure:.2f} months")
print(f"Customer Lifetime Value (CLV): ₹{clv:,.2f}")

ARPU: ₹885.12
Average Customer Lifetime: 76.93 months
Customer Lifetime Value (CLV): ₹68,094.15


## KPI 5: Revenue Lost to Churn

Revenue Lost to Churn represents the monthly recurring revenue associated with customers who have churned.

It is calculated by summing the monthly charges from the latest subscription record of churned customers.

In [22]:
churned_customers = customers[
    customers["customer_status"] == "Churned"
]["customer_id"]

churned_subscriptions = latest_subscriptions[
    latest_subscriptions["customer_id"].isin(churned_customers)
]

revenue_lost_to_churn = (
    churned_subscriptions["monthly_charge_inr"].sum()
)

print("Churned Customers:", len(churned_customers))
print("Churned Customers with Subscription Data:", len(churned_subscriptions))
print(f"Revenue Lost to Churn: ₹{revenue_lost_to_churn:,.2f}")

Churned Customers: 3506
Churned Customers with Subscription Data: 3506
Revenue Lost to Churn: ₹2,404,571.00


## KPI 6: Monthly Recurring Revenue (MRR)

Monthly Recurring Revenue (MRR) represents the total recurring monthly revenue generated from active subscribers.

It is calculated by summing the monthly charges of the latest active subscriptions.

In [24]:
mrr = active_latest["monthly_charge_inr"].sum()

print(f"Active Subscribers: {len(active_latest)}")
print(f"Monthly Recurring Revenue (MRR): ₹{mrr:,.2f}")

Active Subscribers: 15668
Monthly Recurring Revenue (MRR): ₹13,868,137.00


## KPI 7: First Contact Resolution (FCR)

First Contact Resolution (FCR) measures the percentage of resolved support tickets that were resolved during the first contact.

FCR = Tickets Resolved on First Contact/Total Resolved Tickets x 100

Before calculating the KPI, we will inspect the values of the relevant fields to ensure the calculation matches the dataset.

### Step 1: Inspect FCR fields

In [25]:
print("Support Ticket Status Values:")
print(support_tickets["status"].value_counts(dropna=False))

print("\nFirst Contact Resolution Values:")
print(support_tickets["first_contact_resolution"].value_counts(dropna=False))

Support Ticket Status Values:
status
Resolved     18124
Pending       8450
Escalated     2526
Reopened      2390
Name: count, dtype: int64

First Contact Resolution Values:
first_contact_resolution
No     18764
Yes    12726
Name: count, dtype: int64


For FCR, we should count only tickets that are both Resolved and Yes. But, We should not use all 12,726 "Yes" records as the numerator because some could belong to non-resolved tickets.

### Step 2: Calculate FCR

In [26]:
resolved_tickets = support_tickets[
    support_tickets["status"] == "Resolved"
]

fcr_tickets = resolved_tickets[
    resolved_tickets["first_contact_resolution"] == "Yes"
]

fcr = (
    len(fcr_tickets) / len(resolved_tickets)
) * 100

print("Total Resolved Tickets:", len(resolved_tickets))
print("Tickets Resolved on First Contact:", len(fcr_tickets))
print(f"First Contact Resolution (FCR): {fcr:.2f}%")

Total Resolved Tickets: 18124
Tickets Resolved on First Contact: 12726
First Contact Resolution (FCR): 70.22%


## KPI 8: Average Resolution Time

Average Resolution Time measures the average number of hours taken to resolve support tickets.

The calculation uses the `resolution_hours` field for resolved tickets only.

In [27]:
resolved_tickets = support_tickets[
    support_tickets["status"] == "Resolved"
]

average_resolution_time = (
    resolved_tickets["resolution_hours"].mean()
)

print("Resolved Tickets:", len(resolved_tickets))
print(f"Average Resolution Time: {average_resolution_time:.2f} hours")

Resolved Tickets: 18124
Average Resolution Time: 16.04 hours


## KPI 9: Contract Renewal Rate

Contract Renewal Rate measures the percentage of contracts that were renewed among contracts that were due for renewal.

Formula:

Contract Renewal Rate = Contracts Renewed / Contracts Due for Renewal × 100

STEP 1: LOAD AND INSPECT CONTRACTS

In [28]:
contracts = pd.read_csv(
    cleaned_data_path / "contracts.csv"
)

print("Contracts:", contracts.shape)

print("\nContract columns:")
for column in contracts.columns:
    print(column)

Contracts: (10607, 9)

Contract columns:
contract_id
customer_id
plan_id
contract_type
contract_length_months
start_date
end_date
auto_renew
renewal_status


STEP 2: INSPECT RENEWAL STATUS AND DATE

In [29]:
print("Renewal status values:")
print(contracts["renewal_status"].value_counts(dropna=False))

print("\nAuto-renew values:")
print(contracts["auto_renew"].value_counts(dropna=False))

print("\nEnd date sample:")
print(contracts["end_date"].head(10))

Renewal status values:
renewal_status
Renewed         3676
Auto-Renewed    2266
Active          1829
Pending         1360
Not Renewed     1058
Cancelled        418
Name: count, dtype: int64

Auto-renew values:
auto_renew
Yes    5707
No     4900
Name: count, dtype: int64

End date sample:
0    22-12-2015
1    04-05-2017
2    13-10-2015
3    15-06-2019
4    02-01-2026
5    05-03-2026
6    02-02-2020
7    30-06-2026
8    26-04-2025
9    09-07-2023
Name: end_date, dtype: str


we'll treat Renewed + Auto-Renewed as renewed contracts, and contracts whose end_date is on or before the project's reference date (31-03-2026) as contracts due for renewal.

### Step 3: Calculate Contract Renewal Rate

In [30]:
reference_date = pd.Timestamp("2026-03-31")

contracts["end_date_parsed"] = pd.to_datetime(
    contracts["end_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

due_for_renewal = contracts[
    contracts["end_date_parsed"] <= reference_date
]

renewed_contracts = due_for_renewal[
    due_for_renewal["renewal_status"].isin(
        ["Renewed", "Auto-Renewed"]
    )
]

contracts_due = len(due_for_renewal)
contracts_renewed = len(renewed_contracts)

contract_renewal_rate = (
    contracts_renewed / contracts_due
) * 100

print("Contracts Due for Renewal:", contracts_due)
print("Contracts Renewed:", contracts_renewed)
print(f"Contract Renewal Rate: {contract_renewal_rate:.2f}%")

Contracts Due for Renewal: 9651
Contracts Renewed: 5516
Contract Renewal Rate: 57.15%


## KPI 10: AVERAGE CUSTOMER TENURE

WE HAVE ALREADY CALCULATED THIS EARLIER IN STEP 1 OF KPI 4.

AVERAGE CUSTOMER TENURE = 76.93 MONTHS

## FINAL KPI VERIFICATION:

In [31]:
print("NexaTel Phase 3 — Final KPI Summary")
print("=" * 45)

print(f"1. Churn Rate:                {churn_rate:.2f}%")
print(f"2. Retention Rate:            {retention_rate:.2f}%")
print(f"3. ARPU:                      ₹{arpu:,.2f}")
print(f"4. Customer Lifetime Value:   ₹{clv:,.2f}")
print(f"5. Revenue Lost to Churn:     ₹{revenue_lost_to_churn:,.2f}")
print(f"6. Monthly Recurring Revenue: ₹{mrr:,.2f}")
print(f"7. First Contact Resolution:  {fcr:.2f}%")
print(f"8. Average Resolution Time:   {average_resolution_time:.2f} hours")
print(f"9. Contract Renewal Rate:     {contract_renewal_rate:.2f}%")
print(f"10. Average Customer Tenure:  {average_tenure:.2f} months")

NexaTel Phase 3 — Final KPI Summary
1. Churn Rate:                18.45%
2. Retention Rate:            81.55%
3. ARPU:                      ₹885.12
4. Customer Lifetime Value:   ₹68,094.15
5. Revenue Lost to Churn:     ₹2,404,571.00
6. Monthly Recurring Revenue: ₹13,868,137.00
7. First Contact Resolution:  70.22%
8. Average Resolution Time:   16.04 hours
9. Contract Renewal Rate:     57.15%
10. Average Customer Tenure:  76.93 months


### NOW LETS REMOVE THE TEMPORARY CONTRACT COLUMN

In [32]:
if "end_date_parsed" in contracts.columns:
    contracts = contracts.drop(columns=["end_date_parsed"])

print("Temporary columns removed.")
print("Contract columns:")
for column in contracts.columns:
    print(column)

Temporary columns removed.
Contract columns:
contract_id
customer_id
plan_id
contract_type
contract_length_months
start_date
end_date
auto_renew
renewal_status


### REMOVING REMAINING TEMPORARY COLUMNS:

In [33]:
if "start_date_parsed" in subscriptions.columns:
    subscriptions = subscriptions.drop(columns=["start_date_parsed"])

print("Temporary subscription columns removed.")
print("Subscription columns:")
for column in subscriptions.columns:
    print(column)

Temporary subscription columns removed.
Subscription columns:
subscription_id
customer_id
plan_id
start_date
end_date
billing_cycle
monthly_charge_inr
subscription_status
